# Phase57：Phase56実測データからGo2のURDF動画を生成

- 追加学習・物理シミュレーションは行わず、Phase56の記録を再生します。
- Phase56 ZIPをアップロードして実行してください。
- 赤・青のGo2、60秒、10fps。

In [ ]:
%pip -q install pybullet imageio imageio-ffmpeg pillow
from google.colab import files
uploaded=files.upload()
zip_path=next((k for k in uploaded if k.endswith(".zip")),None)
assert zip_path,"Phase56結果ZIPをアップロードしてください"

In [ ]:
from pathlib import Path
import zipfile,csv,io,glob
z=zipfile.ZipFile(zip_path)
assert "joint_poses.csv" in z.namelist() and "joint_name_mapping.csv" in z.namelist()
rows=list(csv.DictReader(io.StringIO(z.read("joint_poses.csv").decode("utf-8-sig"))))
steps={}
for r in rows: steps.setdefault(int(r["step"]),{})[r["robot"]]=r
assert len(steps)==3000 and all(set(v)=={"A","B"} for v in steps.values())
mapping=list(csv.DictReader(io.StringIO(z.read("joint_name_mapping.csv").decode("utf-8-sig"))))
assert len(mapping)==12
# Genesisのインストールは不要。公式リポジトリからGo2のURDFとメッシュを取得します。
import subprocess
assets_repo=Path("/content/genesis_go2_assets")
if not assets_repo.exists():
 subprocess.run(["git","clone","--depth","1","--filter=blob:none","--sparse","https://github.com/Genesis-Embodied-AI/Genesis.git",str(assets_repo)],check=True)
subprocess.run(["git","-C",str(assets_repo),"sparse-checkout","set","genesis/assets/urdf/go2"],check=True)
candidates=list(assets_repo.rglob("go2.urdf"))

try:
 import genesis
 candidates+=list(Path(genesis.__file__).resolve().parent.rglob("go2.urdf"))
except Exception as e: print("Genesis未検出:",e)
for base in (Path("/content"),):
 if base.exists(): candidates += [Path(p) for p in glob.glob(str(base/"**/go2.urdf"),recursive=True)][:15]
candidates=list(dict.fromkeys(p for p in candidates if p.is_file()))
assert candidates,"Go2 URDFが見つかりません。Phase56実行済みColab環境またはGenesisを導入してください"
urdf_path=candidates[0]
print("URDF:",urdf_path,"steps:",len(steps))

In [ ]:
import pybullet as p,numpy as np,imageio.v2 as imageio
from PIL import Image,ImageDraw
cid=p.connect(p.DIRECT)
p.setAdditionalSearchPath(str(urdf_path.parent))
p.createMultiBody(0,p.createCollisionShape(p.GEOM_PLANE))
ids={}; joint_ids={}
for robot in ("A","B"):
 rid=p.loadURDF(str(urdf_path),[0,0,0.42],flags=p.URDF_USE_INERTIA_FROM_FILE)
 ids[robot]=rid
 names={p.getJointInfo(rid,i)[1].decode():i for i in range(p.getNumJoints(rid))}
 assert all(m["joint_name"] in names for m in mapping),"URDF関節名が不一致"
 joint_ids[robot]={m["joint_name"]:names[m["joint_name"]] for m in mapping}
 color=(0.15,0.42,0.96,1) if robot=="A" else (0.95,0.23,0.18,1)
 for j in [-1]+list(range(p.getNumJoints(rid))): p.changeVisualShape(rid,j,rgbaColor=color)
print("関節マッピングOK")

In [ ]:
from IPython.display import display
WIDTH,HEIGHT,FPS=960,540,10
def camera_for_step(step):
 a=steps[step]["A"]; b=steps[step]["B"]
 center=[(float(a["qpos_0"])+float(b["qpos_0"]))/2,(float(a["qpos_1"])+float(b["qpos_1"]))/2,0.22]
 gap=np.hypot(float(a["qpos_0"])-float(b["qpos_0"]),float(a["qpos_1"])-float(b["qpos_1"]))
 return p.computeViewMatrixFromYawPitchRoll(center,max(2.1,min(5.0,2.1+gap*0.65)),35,-25,0,2)
proj=p.computeProjectionMatrixFOV(52,WIDTH/HEIGHT,0.1,30)
def render(step):
 for robot in ("A","B"):
  r=steps[step][robot]; rid=ids[robot]
  xyz=[float(r[f"qpos_{i}"]) for i in range(3)]
  quat=[float(r[f"qpos_{i}"]) for i in range(3,7)]
  p.resetBasePositionAndOrientation(rid,xyz,[quat[1],quat[2],quat[3],quat[0]])
  for m in mapping:
   local=int(m["dofs_idx_local"])
   p.resetJointState(rid,joint_ids[robot][m["joint_name"]],float(r[f"qpos_{local+1}"]))
 arr=np.asarray(p.getCameraImage(WIDTH,HEIGHT,camera_for_step(step),proj,renderer=p.ER_TINY_RENDERER,shadow=0)[2],dtype=np.uint8).reshape(HEIGHT,WIDTH,4)[:,:,:3]
 img=Image.fromarray(arr); draw=ImageDraw.Draw(img)
 draw.rectangle((0,0,WIDTH,42),fill=(17,23,35))
 draw.text((18,13),f"Genesis Phase56 | t={step*0.02:.2f}s | A BLUE / B RED",fill="white")
 return img
display(render(1).resize((640,360)))

In [ ]:
# Phase57C: 1体の姿勢・関節順序・足先位置を静止状態で検証
# 注意：関節値が一致しても、URDFの幾何学的整合性までは保証しません。
from IPython.display import display
import pandas as pd
robot="A"; rid=ids[robot]
links={p.getJointInfo(rid,j)[12].decode():j for j in range(p.getNumJoints(rid))}
foot_links={name:j for name,j in links.items() if "foot" in name.lower()}
print("足先リンク:",foot_links)
assert len(foot_links)>=4, "足先リンクが4つ見つかりません。URDFのリンク名を確認してください"
def inspect_pose(step):
 render(step)
 r=steps[step][robot]
 report=[]
 for m in mapping:
  j=joint_ids[robot][m["joint_name"]]
  info=p.getJointInfo(rid,j)
  q=float(r["qpos_"+str(int(m["dofs_idx_local"])+1)])
  actual=p.getJointState(rid,j)[0]
  low,high=info[8],info[9]
  report.append({"joint":m["joint_name"],"qpos_index":int(m["dofs_idx_local"])+1,
                 "recorded_rad":round(q,3),"bullet_rad":round(actual,3),
                 "limit_low":round(low,3),"limit_high":round(high,3),
                 "outside_limit":bool(low<high and (q<low-0.01 or q>high+0.01))})
 feet={name:round(p.getLinkState(rid,j,computeForwardKinematics=True)[0][2],3)
       for name,j in foot_links.items()}
 print("STEP",step,"time",round(step*0.02,2),"s","base_z",round(float(r["qpos_2"]),3),
       "foot_world_z",feet)
 assert all(abs(v["recorded_rad"]-v["bullet_rad"])<0.002 for v in report)
 display(pd.DataFrame(report))
 display(render(step).resize((800,450)))
for step in (1,500,1500):
 inspect_pose(step)
print("診断完了：足先高さ、関節可動範囲、静止姿勢を確認してください")


In [ ]:
# 5秒の脚動作確認（60秒のレンダリング前に実行）
for robot in ("A","B"):
 deltas=[]
 for m in mapping:
  key="qpos_"+str(int(m["dofs_idx_local"])+1)
  values=[float(steps[k][robot][key]) for k in range(1,251)]
  delta=max(values)-min(values)
  deltas.append(delta)
  print(robot,m["joint_name"],round(delta,3),"rad")
 assert max(deltas)>0.05,"関節角の変化が小さすぎます"
preview="phase57_leg_test_5s.mp4"
with imageio.get_writer(preview,fps=20,codec="libx264",quality=8,macro_block_size=16) as w:
 for k in range(100): w.append_data(np.asarray(render(1+2*k)))
from IPython.display import Video,display
display(Video(preview,embed=True,width=720))

# URDFへの反映を数値で照合（Aの12関節）
render(1)
for m in mapping:
 j=joint_ids["A"][m["joint_name"]]
 recorded=float(steps[1]["A"]["qpos_"+str(int(m["dofs_idx_local"])+1)])
 actual=p.getJointState(ids["A"],j)[0]
 assert abs(recorded-actual)<1e-5,(m["joint_name"],recorded,actual)
print("12関節の記録値とPyBullet反映値が一致")
# 脚の動きを見やすくするため、同じ5秒を半速で再生
slow="phase57_leg_test_5s_half_speed.mp4"
with imageio.get_writer(slow,fps=10,codec="libx264",quality=8,macro_block_size=16) as w:
 for k in range(100): w.append_data(np.asarray(render(1+2*k)))
display(Video(slow,embed=True,width=720))


In [ ]:
out="phase57_go2_urdf_60s.mp4"
with imageio.get_writer(out,fps=FPS,codec="libx264",quality=8,macro_block_size=16) as w:
 for k in range(600):
  step=1+k*5
  w.append_data(np.asarray(render(step)))
  if k%100==0: print("描画",k,"/600")
print("生成完了",out,Path(out).stat().st_size)
files.download(out)